# HAM10000 — Stage 5: Grad-CAM

**Goal:** Visualise which image regions drive each model's predictions using Grad-CAM.
This is the interpretability stage — it validates (or flags) whether models look at
lesion structure or spurious background features.

**What Grad-CAM shows mathematically:**  
Grad-CAM weights each feature map by the gradient of the target class score w.r.t.
that map, then takes a weighted sum + ReLU:  
`L = ReLU(Σ_k α_k · A^k)` where `α_k = (1/Z) Σ ∂y^c/∂A^k_{ij}`  
Result: a coarse heatmap showing regions that **positively** contributed to the predicted class.

**Target layers:**
| Model | Target Layer | Spatial Size | Why |
|---|---|---|---|
| Baseline CNN | `features[3].block[0]` | 28×28 | Last conv before GAP — semantic features, good resolution |
| ResNet18 | `layer4[1].conv2` | 7×7 | Last conv in last residual block — standard choice |
| EfficientNet-B0 | `features[8][0]` | 7×7 | Last MBConv block — highest-level features before pooling |

## Cell 1 — Clone / update repo and set up paths

In [ ]:
import os, sys, subprocess

REPO_URL  = "https://github.com/Dev252001/HAM10000.git"
REPO_DIR  = "/content/ham10000-classifier"
SRC_DIR   = os.path.join(REPO_DIR, "src")
DATA_DIR  = os.path.join(REPO_DIR, "data")
OUT_DIR   = os.path.join(REPO_DIR, "outputs")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print(f"src/ on path: {SRC_DIR}")

## Cell 2 — Install dependencies (includes grad-cam)

In [ ]:
!pip install -q \
  "numpy>=2.0" \
  "pandas>=2.2.2" \
  "Pillow>=10.4.0" \
  "scikit-learn>=1.5.0" \
  "matplotlib>=3.9.0" \
  "grad-cam>=1.5.0" \
  "kaggle>=1.6.14"
print("Dependencies ready.")

## Cell 3 — GPU + imports

In [ ]:
import torch
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Cell 4 — Dataset (Google Drive cache + Kaggle fallback)

Loads data from Drive if available (instant), otherwise downloads from Kaggle once and saves to Drive.

In [ ]:
import os, json, shutil
from google.colab import drive
from data_loader import download_dataset

DRIVE_DATA = "/content/drive/MyDrive/HAM10000_data"
DATA_DIR   = "/content/ham10000-classifier/data"

drive.mount('/content/drive')

if os.path.exists(os.path.join(DRIVE_DATA, 'HAM10000_metadata.csv')):
    print('Dataset found on Drive — copying to /content/ ...')
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    shutil.copytree(DRIVE_DATA, DATA_DIR)
    print('Done ✓')
else:
    print('Dataset not on Drive — downloading from Kaggle (one-time, ~10 min)...')
    from google.colab import files as colab_files
    uploaded = colab_files.upload()
    uploaded_name = list(uploaded.keys())[0]
    creds = json.loads(uploaded[uploaded_name])
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/kaggle.json', 'w') as f:
        json.dump(creds, f)
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print(f'Kaggle credentials configured (user: {creds["username"]}).')
    download_dataset(DATA_DIR)
    print('Saving to Google Drive for future sessions...')
    shutil.copytree(DATA_DIR, DRIVE_DATA)
    print('Saved to Drive ✓ — future sessions will skip the Kaggle download')

from data_loader import load_metadata, CLASSES, LABEL_MAP
from preprocessing import make_splits, get_transforms

df = load_metadata(DATA_DIR)
_, _, test_df = make_splits(df)   # same splits as training — random_state=42
test_tf = get_transforms('val')   # no augmentation for Grad-CAM

print(f"Test set: {len(test_df)} images")

## Cell 5 — Load trained model checkpoints

Loads all three checkpoints. If a checkpoint is missing, that model is skipped.
**At minimum, the best transfer model checkpoint must be present for the comparisons.**

In [ ]:
from models.baseline_cnn import build_baseline_cnn
from models.transfer_models import build_resnet18, build_efficientnet_b0

loaded_models = {}

checkpoints = {
    "baseline"        : os.path.join(OUT_DIR, "models", "baseline_cnn_best.pt"),
    "resnet18"        : os.path.join(OUT_DIR, "models", "resnet18_best.pt"),
    "efficientnet_b0" : os.path.join(OUT_DIR, "models", "efficientnet_b0_best.pt"),
}

builders = {
    "baseline"        : lambda: build_baseline_cnn(num_classes=7)[0] if isinstance(build_baseline_cnn(num_classes=7), tuple) else build_baseline_cnn(num_classes=7),
    "resnet18"        : lambda: build_resnet18(num_classes=7)[0],
    "efficientnet_b0" : lambda: build_efficientnet_b0(num_classes=7)[0],
}

for name, ckpt_path in checkpoints.items():
    if os.path.exists(ckpt_path):
        model = builders[name]()
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        model = model.to(device)
        model.eval()
        loaded_models[name] = model
        print(f"✓ {name:20s} loaded from {ckpt_path}")
    else:
        print(f"⚠️  {name:20s} checkpoint NOT found — skipping")

print(f"\nModels available for Grad-CAM: {list(loaded_models.keys())}")

## Cell 6 — Helper: get predictions on test set

Builds a lookup of (image_tensor, true_label, pred_label) for every test image
so we can easily find correct/incorrect predictions per class.

In [ ]:
from PIL import Image as PILImage

def get_predictions(model, test_df, transform, device, n_per_class=10):
    """
    Run model on a sample of test images and return a list of result dicts.
    Samples up to n_per_class images per class to keep it manageable.
    """
    model.eval()
    results = []

    # Sample up to n_per_class per class
    sampled = test_df.groupby('dx').apply(
        lambda g: g.sample(min(len(g), n_per_class), random_state=42)
    ).reset_index(drop=True)

    with torch.no_grad():
        for _, row in sampled.iterrows():
            img   = PILImage.open(row['filepath']).convert('RGB')
            t     = transform(img)
            out   = model(t.unsqueeze(0).to(device))
            pred  = out.argmax(dim=1).item()
            results.append({
                'image_tensor' : t,
                'true_label'   : int(row['class_idx']),
                'pred_label'   : pred,
                'image_id'     : row['image_id'],
                'dx'           : row['dx'],
            })
    return results

# Use best available transfer model for primary comparisons
if not loaded_models:
    raise RuntimeError(
        "No model checkpoints found in outputs/models/.\n"
        "You need to run Stage 3 and/or Stage 4 first to generate checkpoints.\n"
        "Checkpoints needed: resnet18_best.pt or efficientnet_b0_best.pt"
    )

BEST_MODEL_NAME = "efficientnet_b0" if "efficientnet_b0" in loaded_models else \
                  "resnet18"        if "resnet18"        in loaded_models else \
                  "baseline"
print(f"Primary model for Grad-CAM: {BEST_MODEL_NAME}")

best_model = loaded_models[BEST_MODEL_NAME]
predictions = get_predictions(best_model, test_df, test_tf, device, n_per_class=10)
print(f"Collected {len(predictions)} predictions")

## Cell 7 — Grad-CAM: Correct predictions on malignant classes

Shows 2–3 correctly classified images per malignant class (mel, bcc, akiec).

**Sanity check:** The heatmap should focus on the **lesion** (centre/irregular region),
not on background skin, hair, or ruler markings.
If heatmaps land on background → the model may have learned a spurious shortcut → flag it.

In [ ]:
import os
from gradcam import gradcam_grid
from data_loader import MALIGNANT_CLASSES, CLASS_TO_IDX

MALIGNANT_IDX = {CLASS_TO_IDX[c] for c in MALIGNANT_CLASSES}

# Correct predictions on malignant classes
correct_malignant = [
    p for p in predictions
    if p['true_label'] in MALIGNANT_IDX and p['pred_label'] == p['true_label']
]

# Pick up to 2 per malignant class
samples_correct = []
for cls_idx in sorted(MALIGNANT_IDX):
    cls_samples = [p for p in correct_malignant if p['true_label'] == cls_idx]
    samples_correct.extend(cls_samples[:2])

print(f"Selected {len(samples_correct)} correctly classified malignant images")

if samples_correct:
    gradcam_grid(
        model      = best_model,
        model_name = BEST_MODEL_NAME,
        samples    = samples_correct,
        title      = f"{BEST_MODEL_NAME} — Grad-CAM: Correct malignant predictions\n"
                     "Heatmap should focus on the lesion (not background skin or hair)",
        save_path  = os.path.join(OUT_DIR, "figures", "gradcam_correct_malignant.png"),
    )
else:
    print("⚠️  No correctly classified malignant images found. Check model checkpoint.")

## Cell 8 — Grad-CAM: Incorrect predictions (malignant → benign)

Shows misclassified images where a **malignant class was predicted as benign**.
This is the clinically worst error type.

**What to look for in the heatmap:**
- Heatmap on hair/ruler/dark corner → model distracted by image artifact
- Heatmap on lesion but wrong class → genuinely ambiguous features
- Heatmap scattered/diffuse → model is uncertain, no dominant feature

In [ ]:
from data_loader import CLASS_TO_IDX, CLASSES

BENIGN_IDX = {CLASS_TO_IDX[c] for c in CLASSES if c not in MALIGNANT_CLASSES}

# Malignant → benign misclassifications (worst clinical error)
false_negatives = [
    p for p in predictions
    if p['true_label'] in MALIGNANT_IDX and p['pred_label'] in BENIGN_IDX
]

# Also include any malignant → wrong malignant misclassifications
false_negative_ids = {p['image_id'] for p in false_negatives}
wrong_malignant = [
    p for p in predictions
    if p['true_label'] in MALIGNANT_IDX
    and p['pred_label'] != p['true_label']
    and p['image_id'] not in false_negative_ids
]

# Prefer false negatives; fall back to wrong malignant if not enough
samples_incorrect = (false_negatives + wrong_malignant)[:6]

print(f"False negatives (malignant→benign): {len(false_negatives)}")
print(f"Other malignant misclassifications: {len(wrong_malignant)}")
print(f"Selected {len(samples_incorrect)} for Grad-CAM")

if samples_incorrect:
    gradcam_grid(
        model      = best_model,
        model_name = BEST_MODEL_NAME,
        samples    = samples_incorrect,
        title      = f"{BEST_MODEL_NAME} — Grad-CAM: Misclassified malignant images\n"
                     "Red title = wrong prediction. Check if heatmap lands on artifact vs lesion.",
        save_path  = os.path.join(OUT_DIR, "figures", "gradcam_incorrect_malignant.png"),
    )
else:
    print("⚠️  No malignant misclassifications found in the sampled test images.")
    print("   Try increasing n_per_class in Cell 6 or use the full test set.")

## Cell 9 — Side-by-side: Baseline CNN vs. best transfer model

For the same images, shows what each model "looks at".
This directly visualises the interpretability improvement from transfer learning.

In [ ]:
from gradcam import gradcam_model_comparison

# Pick 3 representative images: one per malignant class if possible
comparison_samples = []
for cls_idx in sorted(MALIGNANT_IDX):
    # Prefer correctly classified by best model
    cls_correct = [p for p in predictions
                   if p['true_label'] == cls_idx and p['pred_label'] == cls_idx]
    if cls_correct:
        comparison_samples.append(cls_correct[0])
    elif any(p['true_label'] == cls_idx for p in predictions):
        comparison_samples.append(next(p for p in predictions if p['true_label'] == cls_idx))

comparison_samples = comparison_samples[:3]  # max 3 for readability

# Build models dict — only include models that are loaded
comparison_models = {}
if "baseline" in loaded_models:
    comparison_models["baseline"] = loaded_models["baseline"]
comparison_models[BEST_MODEL_NAME] = best_model

print(f"Comparing: {list(comparison_models.keys())}")
print(f"Images: {[s['image_id'] for s in comparison_samples]}")

if len(comparison_models) >= 2 and comparison_samples:
    gradcam_model_comparison(
        models_dict = comparison_models,
        samples     = comparison_samples,
        title       = "Grad-CAM comparison: Baseline CNN vs. Transfer model\n"
                      "Columns: Original | Baseline | Transfer",
        save_path   = os.path.join(OUT_DIR, "figures", "gradcam_model_comparison.png"),
    )
else:
    print("Need at least 2 loaded models for comparison.")
    print("If baseline checkpoint is missing, run Stage 3 first.")

## Cell 10 — Heatmap sanity check summary

After reviewing the Grad-CAM figures above, assess each heatmap against
the criteria below and record your observations.

In [ ]:
print("="*65)
print("GRAD-CAM SANITY CHECK — fill in after reviewing the figures")
print("="*65)
print()
print("For each Grad-CAM figure, check:")
print()
print("1. CORRECT MALIGNANT PREDICTIONS (gradcam_correct_malignant.png)")
print("   ✓ PASS: heatmap centres on the lesion (irregular/dark/coloured region)")
print("   ✗ FAIL: heatmap on background skin, hair, ruler, dark corner")
print("   → If FAIL: model may have a spurious shortcut — note for writeup")
print()
print("2. INCORRECT MALIGNANT PREDICTIONS (gradcam_incorrect_malignant.png)")
print("   Look for:")
print("   - Hair/artifact focus → model distracted by known HAM10000 artifacts")
print("   - Lesion focus but wrong class → genuinely ambiguous features")
print("   - Diffuse/scattered heatmap → model uncertain, no dominant cue")
print()
print("3. BASELINE vs TRANSFER (gradcam_model_comparison.png)")
print("   ✓ EXPECTED: transfer model focuses more tightly on lesion")
print("   ✓ EXPECTED: baseline may be more diffuse or focus on background")
print("   → Tighter lesion focus = transfer learning learned more relevant features")
print()
print("Known HAM10000 artifacts to watch for:")
print("  - Hair/black fibres crossing the lesion")
print("  - Ruler/scale markings at image edges")
print("  - Dark circular vignette at image corners (dermatoscope lens boundary)")
print("  - Air bubbles (bright circular spots)")

## Stage 5 complete ✓

**Before moving to Stage 6 (final writeup), verify all boxes below:**

- [ ] `gradcam_correct_malignant.png` saved — heatmaps visible, overlaid on lesions
- [ ] `gradcam_incorrect_malignant.png` saved — at least 2–3 misclassification examples
- [ ] `gradcam_model_comparison.png` saved — columns show baseline vs. transfer model
- [ ] Correct prediction heatmaps land on the lesion, not background (flag if not)
- [ ] Misclassification heatmaps reviewed — note whether artifact or ambiguous features caused the error
- [ ] Transfer model heatmaps are tighter/more lesion-focused than baseline (expected, flag if reversed)

**Share screenshots of the Grad-CAM figures before Stage 6 starts.**

---

### What was built and why

| Component | Choice | Reason |
|---|---|---|
| CAM method | Grad-CAM (not GradCAM++ or EigenCAM) | Standard, well-understood, easy to explain mathematically in interview |
| Library | pytorch-grad-cam | Battle-tested, handles BatchNorm/inplace-ReLU edge cases correctly |
| Baseline target layer | `features[3].block[0]` (28×28) | Last conv before GAP — semantic features with highest resolution |
| ResNet18 target layer | `layer4[1].conv2` (7×7) | Last conv in last residual block — conventional choice in literature |
| EfficientNet target layer | `features[8][0]` (7×7) | Last MBConv block — highest-level features before global pooling |
| Targeting predicted class | yes | Shows what drove THIS prediction (including wrong ones) |

---

### Likely interview questions on this stage

**Q1: What does Grad-CAM actually show mathematically, and what are its limitations?**  
Grad-CAM computes the gradient of the target class score with respect to each feature map in the target layer, averages spatially to get per-channel importance weights (α_k), then takes a ReLU-weighted sum of the feature maps. The ReLU clips negative contributions (regions that suppressed the class score). Limitations: the resolution is limited by the target layer's spatial size (7×7 for ResNet18, upsampled to 224×224 — so it's inherently coarse), it only shows one class at a time, and it can be sensitive to the choice of target layer.

**Q2: Why did you target those specific layers for each architecture?**  
The target layer must be a convolutional layer that is (a) late enough to have learned semantically meaningful features, and (b) early enough to still have spatial resolution greater than 1×1. The layer immediately before Global Average Pooling is usually ideal — it has the highest-level features and still has a spatial map. For ResNet18 that's `layer4[1].conv2` (7×7), for EfficientNet-B0 it's `features[8][0]` (7×7), and for our baseline we target before the last MaxPool to get 28×28 resolution.

**Q3: What did the misclassification heatmaps reveal about model failure modes?**  
This answer should reference your actual results — the common patterns in HAM10000 are: (1) heatmaps on hair crossing the lesion (hair is a known artifact — many HAM10000 images have black fibres over lesions), (2) heatmaps on the dark vignette at image corners from the dermatoscope lens, (3) diffuse heatmaps when the lesion has low contrast against surrounding skin. If the model focused on lesion structure even for wrong predictions, the error is likely due to genuinely ambiguous features rather than spurious shortcuts.